# FiberPulse AI: Day 2 notebook (Phase 1, synthetic data)

Explore the data we generated today. Everything is invented. Run the cells top to bottom.
Contents: 1) setup, 2) tables, 3) scores and flags, 4) a case file (paradox vs confirmed), 5) why `eval/` is hidden,
6) the policy oracle, 7) a naive retrieval preview of the Outcomes KB.

In [ ]:
import json, sqlite3, subprocess, sys
from pathlib import Path
import pandas as pd

root = Path.cwd()
while root != root.parent and not (root / "pyproject.toml").exists():
    root = root.parent
sys.path.insert(0, str(root / "src"))
DATA = root / "data" / "synthetic"
if not (DATA / "lakehouse.db").exists():
    subprocess.run([sys.executable, "-m", "fiberpulse.synth.build", "--n-accounts", "20000", "--seed", "42",
                    "--out", str(DATA)], cwd=root, check=True)
con = sqlite3.connect(DATA / "lakehouse.db")
def q(sql, params=()):
    return pd.read_sql_query(sql, con, params=params)
print("Project root:", root)

## 2. Tables in the lakehouse
Row counts per table. The scoring date is 2026-10-01; the model saw data only up to 2026-08-31.

In [ ]:
tables = [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
display(pd.DataFrame({"table": tables, "rows": [con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in tables]}))

In [ ]:
display(q("SELECT segment, COUNT(*) AS accounts, ROUND(AVG(mrc_aed),0) AS avg_mrc, ROUND(AVG(tenure_months),1) AS avg_tenure FROM customer_360 GROUP BY segment"))

## 3. Scores and flags
The top 6.25% per signal are flagged (1,250 of 20,000). Flagged scores start at 0.70.

In [ ]:
display(q("""SELECT signal_type, COUNT(*) AS n, SUM(flagged) AS flagged,
       ROUND(MIN(CASE WHEN flagged=1 THEN score END),3) AS min_flagged,
       ROUND(MAX(score),3) AS max_score FROM model_scores GROUP BY signal_type"""))

## 4. A case file
Agent 3's job in miniature: gather what we know about one flagged account. The helper below shows the account, recent
tickets, calls and outside signals. Compare a **paradox** (the score is stale) with a **confirmed** case.

In [ ]:
def case_file(account_id, signal):
    print(q("SELECT * FROM customer_360 WHERE account_id=?", (account_id,)).T.to_string(header=False))
    print("\nScore:", q("SELECT score, flagged FROM model_scores WHERE account_id=? AND signal_type=?", (account_id, signal)).to_dict("records"))
    print("\nTickets since 2026-06-03:")
    display(q("SELECT opened_date, closed_date, category, severity, status, csat FROM sr_complaint WHERE account_id=? AND opened_date>='2026-06-03' ORDER BY opened_date", (account_id,)))
    print("Calls since 2026-06-03:")
    display(q("SELECT call_date, direction, intent, sentiment, summary FROM call_intent_signals WHERE account_id=? AND call_date>='2026-06-03' ORDER BY call_date", (account_id,)))
    print("External signals:")
    display(q("SELECT signal_date, source, polarity, signal_text FROM external_enrichment WHERE account_id=? ORDER BY signal_date", (account_id,)))
    print("Last 3 bills:")
    display(q("SELECT * FROM bills WHERE account_id=? ORDER BY 2 DESC LIMIT 3", (account_id,)))

# We peek at eval/ here ONLY to pick examples. Agents must never do this.
ev = json.loads((DATA / "eval" / "eval_set.json").read_text(encoding="utf-8"))
paradox = next(e for e in ev if e["expected_verdict"] == "paradox" and e["signal_type"] == "churn")
confirmed = next(e for e in ev if e["expected_verdict"] == "confirmed" and e["signal_type"] == "churn")
print("PARADOX example:", paradox["account_id"], "| archetype:", paradox["archetype"], "| score:", paradox["score"])
case_file(paradox["account_id"], "churn")

In [ ]:
print("CONFIRMED example:", confirmed["account_id"], "| archetype:", confirmed["archetype"], "| score:", confirmed["score"])
case_file(confirmed["account_id"], "churn")

## 5. Why `eval/` is hidden
Both accounts above have high scores. What separates them is in the **evidence** (fresh events), and the true answer
sits in `eval/`. If an agent could read that folder, our tests would measure nothing.

In [ ]:
print("Lakehouse columns that look like truth:",
      [c for t in tables for c in [r[1] for r in con.execute(f"PRAGMA table_info({t})")]
       if any(w in c for w in ("true_", "archetype", "expected", "planted", "difficulty"))])
print("Files in eval/:", sorted(p.name for p in (DATA / "eval").iterdir()))
print("Mean true 3-month probability  confirmed:",
      round(sum(e["true_probability_3m"] for e in ev if e["expected_verdict"] == "confirmed") / 25, 3),
      " paradox:", round(sum(e["true_probability_3m"] for e in ev if e["expected_verdict"] == "paradox") / 25, 3))

## 6. The policy oracle
Agent 6 will be graded against this deterministic function. Try your own scenarios.

In [ ]:
from fiberpulse.synth.catalog import OFFERS, OFFER_BY_ID, AccountFacts, evaluate_offer
print(pd.DataFrame([{"offer": o.offer_id, "name": o.name, "type": o.offer_type if hasattr(o, "offer_type") else "",
                     "discount_pct": o.discount_pct, "margin": round(o.gross_margin, 3)} for o in OFFERS]).to_string(index=False))

facts = AccountFacts(segment="Small", tenure_months=24, marketing_opt_in=False, open_high_ticket=True,
                     late_or_unpaid_last3=0, days_since_last_retention_offer=None,
                     days_since_last_upsell_offer=None, active_addons=frozenset())
for oid in ("UP-04", "UP-10", "RET-03"):
    print(oid, "->", evaluate_offer(OFFER_BY_ID[oid], facts) or "passes")

## 7. RAG preview: naive keyword retrieval over the Outcomes KB
Real RAG (Phase 6) uses embeddings. This crude word-overlap version shows the idea and where it fails.

In [ ]:
import re
kb = [json.loads(l) for l in (DATA / "knowledge" / "outcomes_kb.jsonl").read_text(encoding="utf-8").splitlines()]
tok = lambda s: set(re.findall(r"[a-z]+", s.lower()))
def search(query, signal=None, k=3):
    qt = tok(query)
    pool = [c for c in kb if signal is None or c["signal_type"] == signal]
    scored = sorted(pool, key=lambda c: -len(qt & tok(c["text"])) / (len(tok(c["text"])) ** 0.5))
    return scored[:k]
for c in search("open complaints exit terms competitor promotion", "churn"):
    print(c["case_id"], c["signal_validity"], "|", c["outcome"], "|", c["lesson"][:90])
print("\nKB validity mix:", pd.Series([c["signal_validity"] for c in kb]).value_counts().to_dict())

## Next
Phase 2: train real churn and acquisition models (XGBoost, MLflow) on features up to 2026-08-31 and replace the
simulated scores. Then Phase 3: Agent 3 plus a minimal LangGraph.